# 01 — Capa Bronze: ingesta de sismos y maestros
**Sismos** (carpeta `landing/sismos/<evento>/` del Volume): un archivo `.txt` de CISMID por estación. Es la fuente que llega cada vez que ocurre un sismo.
**Maestros** (`landing/maestros/`): Vs30, límites, microzonificación, Open Buildings y su altura 2.5D, muestra de STEAD, Norma E.030, censo, OSM, Tainan, Manta, BSSA14 y fragilidad GEM.

Reglas de Bronze: no se limpia nada; se agregan `_source_file`, `_evento_carga` (o `_fuente` en los maestros) e `_ingested_at`; la carga de sismos es **idempotente**: volver a ejecutar un sismo lo reemplaza, no lo duplica.

Los parámetros `fecha_desde` y `fecha_hasta` los entrega el job y eligen los sismos por su fecha. Los maestros se cargan la primera vez o con `recargar_maestros = true`.

## 1. Setup — imports

In [ ]:
%pip install -q "numpy<2" geopandas rasterio h5py pypdf pyproj shapely seaborn

In [ ]:
# Reinicia Python para que el driver y los workers usen las mismas librerías.
dbutils.library.restartPython()

In [ ]:
# Librerías que usa el notebook.
import json
import re
import zipfile
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import rasterio
import shapely
from pypdf import PdfReader
from rasterio.windows import from_bounds

from pyspark.sql import functions as F, types as T

## 2. Catálogo, esquema y parámetros

In [ ]:
# Catálogo y esquema; las fechas y la recarga de maestros llegan como parámetros del job.
catalogo = "grupo06proyectofinal"
esquema = "sismos"
volumen = f"/Volumes/{catalogo}/{esquema}"

spark.sql(f"USE CATALOG {catalogo}")
spark.sql(f"USE SCHEMA {esquema}")
spark.conf.set("spark.sql.session.timeZone", "America/Lima")

dbutils.widgets.text("fecha_desde", "2007-01-01")
dbutils.widgets.text("fecha_hasta", "2025-12-31")
dbutils.widgets.text("recargar_maestros", "false")
fecha_desde, fecha_hasta = dbutils.widgets.get("fecha_desde"), dbutils.widgets.get("fecha_hasta")
recargar_maestros = dbutils.widgets.get("recargar_maestros").lower() == "true"

In [ ]:
# Área de estudio y constantes del procesamiento de señales.
PROVINCIAS_ESTUDIO = ["LIMA", "CALLAO"]
DISTRITOS_ESTUDIO = {"150104": "BARRANCO", "150122": "MIRAFLORES", "150130": "SAN BORJA", "150131": "SAN ISIDRO",
                     "150140": "SANTIAGO DE SURCO", "150141": "SURQUILLO"}
ZONA_E030_LIMA = 4
EPSG_UTM_LIMA = 32718
SEED = 700
FILTRO_HZ = (0.1, 25.0)
AMORTIGUAMIENTO = 0.05
PERIODOS = np.round(np.geomspace(0.05, 4.0, 50), 4)
G_CM_S2 = 980.665

## 3. Funciones de escritura y lectura

In [ ]:
# Pasa una tabla pandas pequeña a Spark, con los nulos bien tratados.
def a_spark(pdf, esquema=None):
    pdf = pdf.copy()
    for c in pdf.columns:
        if isinstance(pdf[c].dtype, (pd.StringDtype, pd.ArrowDtype)):
            pdf[c] = pdf[c].astype(object)
        if pdf[c].dtype == object:
            pdf[c] = pdf[c].map(lambda v: v if v is None or isinstance(v, (str, bytes, list, dict, np.ndarray))
                                else (None if pd.isna(v) else str(v)))
    return spark.createDataFrame(pdf, esquema) if esquema is not None else spark.createDataFrame(pdf)


# Guarda un DataFrame como tabla Delta y cuenta sus filas.
def escribir(df, tabla, particion=None, modo="overwrite", contar=True):
    escritor = df.write.format("delta").mode(modo)
    if modo == "overwrite":
        escritor = escritor.option("overwriteSchema", "true")
    if particion:
        escritor = escritor.partitionBy(particion)
    escritor.saveAsTable(tabla)
    if contar:
        n = spark.table(tabla).count()
        print(f"  {tabla}: {n:,} filas")
        return n


# Filas y tamaño de las tablas con un prefijo.
def inventario(prefijo):
    filas = []
    for t in spark.sql("SHOW TABLES").collect():
        if t.isTemporary or not t.tableName.startswith(prefijo):
            continue
        d = spark.sql(f"DESCRIBE DETAIL {t.tableName}").first()
        filas.append((t.tableName, spark.table(t.tableName).count(), round((d["sizeInBytes"] or 0) / 1e6, 1)))
    return spark.createDataFrame(filas, "tabla string, filas long, mb double")

In [ ]:
# Agrega fuente, archivo de origen y fecha de ingesta.
def con_metadatos(df, fuente, archivo=None):
    if archivo is not None or "_source_file" not in df.columns:
        df = df.withColumn("_source_file", F.lit(None if archivo is None else str(archivo)).cast("string"))
    return df.withColumn("_fuente", F.lit(fuente)).withColumn("_ingested_at", F.current_timestamp())


# Limpia nombres de columnas para Delta y evita repetidos.
def nombres_validos(columnas):
    vistos, salida = {}, []
    for c in columnas:
        nombre = re.sub(r"[^0-9A-Za-z_]+", "_", str(c).strip()).strip("_") or "columna"
        vistos[nombre] = vistos.get(nombre, 0) + 1
        salida.append(nombre if vistos[nombre] == 1 else f"{nombre}_{vistos[nombre]}")
    return salida


# Guarda en bronze todas las columnas como texto, con nombres válidos.
def subir_crudo(df, tabla, fuente, archivo=None):
    datos = [c for c in df.columns if c != "_source_file"]
    columnas = [F.col(f"`{c}`").cast("string").alias(n) for c, n in zip(datos, nombres_validos(datos))]
    extra = ["_source_file"] if "_source_file" in df.columns else []
    return escribir(con_metadatos(df.select(*columnas, *extra), fuente, archivo), f"bronze_{tabla}")

In [ ]:
# Quita el prefijo dbfs: de una ruta.
def sin_dbfs(columna):
    return F.regexp_replace(columna, "^dbfs:", "")


# Lista archivos de una carpeta con su tamaño y fecha.
def listar(carpeta, patron="*", recursivo=False):
    return (spark.read.format("binaryFile").option("pathGlobFilter", patron).option("recursiveFileLookup", str(recursivo).lower())
            .load(str(carpeta)).select(sin_dbfs(F.col("path")).alias("ruta"), "length", "modificationTime"))


# Reparte los archivos entre los workers y aplica una función de lectura a cada uno.
def leer_archivos(carpeta, patron, leer, esquema, recursivo=False):
    rutas = listar(carpeta, patron, recursivo).select("ruta")
    columnas = spark.createDataFrame([], esquema).columns

    # Aplica la función de lectura a cada archivo del lote.
    def aplicar(lotes):
        for pdf in lotes:
            for ruta in pdf["ruta"]:
                salida = leer(ruta)
                if len(salida):
                    yield salida.reindex(columns=columnas)

    return rutas.repartition(max(rutas.count(), 1)).mapInPandas(aplicar, esquema)


# Lee CSV con Spark, con tipos compatibles y el archivo de origen.
def leer_csv(ruta, como_texto=(), todo_texto=False, **opciones):
    opciones = {"header": True, "escape": '"', **opciones}
    if todo_texto:
        df = spark.read.options(**opciones).csv(str(ruta))
    else:
        inferido = spark.read.options(inferSchema=True, **opciones).csv(str(ruta)).schema

        # Tipo final de una columna del CSV.
        def tipo(c):
            if c.name in como_texto or isinstance(c.dataType, (T.TimestampType, T.TimestampNTZType, T.DateType)):
                return T.StringType()
            return T.LongType() if isinstance(c.dataType, T.IntegerType) else c.dataType

        df = spark.read.options(**opciones).schema(T.StructType([T.StructField(c.name, tipo(c)) for c in inferido])).csv(str(ruta))
    return df.withColumn("_source_file", sin_dbfs(F.col("_metadata.file_path")))


# Convierte geometrías GeoJSON en WKT.
@F.pandas_udf("string")
def geojson_a_wkt(geometria: pd.Series) -> pd.Series:
    return pd.Series(shapely.to_wkt(shapely.from_geojson(geometria.to_numpy(dtype=object))), index=geometria.index)


# Caja envolvente de cada geometría WKT.
@F.pandas_udf("xmin double, ymin double, xmax double, ymax double")
def limites_wkt(wkt: pd.Series) -> pd.DataFrame:
    return pd.DataFrame(shapely.bounds(shapely.from_wkt(wkt.to_numpy(dtype=object))), columns=["xmin", "ymin", "xmax", "ymax"], index=wkt.index)


# Lee un GeoJSON con Spark: propiedades como columnas y geometría en WKT.
def leer_geojson(ruta):
    inferido = spark.read.option("multiLine", True).json(str(ruta)).schema
    propiedades = inferido["features"].dataType.elementType["properties"].dataType
    esquema = T.StructType([T.StructField("features", T.ArrayType(T.StructType([
        T.StructField("properties", propiedades), T.StructField("geometry", T.StringType())])))])
    return (spark.read.schema(esquema).option("multiLine", True).json(str(ruta))
            .select(F.explode("features").alias("f"))
            .select("f.properties.*", geojson_a_wkt(F.col("f.geometry")).alias("geometry_wkt")))

## 4. Sismos que llegaron al Volume
Cada carpeta de `landing/sismos/` es un sismo; su nombre empieza con la fecha. Se cargan los que caen entre `fecha_desde` y `fecha_hasta`.

In [ ]:
# Lista los sismos del Volume dentro del rango de fechas.
landing = f"{volumen}/landing"
MAESTROS = Path(f"{landing}/maestros")
dbutils.fs.mkdirs(f"{landing}/sismos")
eventos = sorted(e.name.rstrip("/") for e in dbutils.fs.ls(f"{landing}/sismos") if fecha_desde <= e.name[:10] <= fecha_hasta)
print("Sismos a procesar:", eventos)

In [ ]:
ESQUEMA_CISMID = ("evento string, archivo_estacion string, bytes long, codificacion string, contenido string, "
                  "_source_file string, _evento_carga string")


# Decodifica cada archivo como UTF-8; si falla, como Latin-1.
def decodificar(lotes):
    for pdf in lotes:
        textos, codigos = [], []
        for datos in pdf["content"]:
            try:
                textos.append(bytes(datos).decode("utf-8"))
                codigos.append("utf-8")
            except UnicodeDecodeError:
                textos.append(bytes(datos).decode("latin-1"))
                codigos.append("latin-1")
        yield pd.DataFrame({"evento": pdf["evento"], "archivo_estacion": pdf["archivo_estacion"], "bytes": pdf["length"], "codificacion": codigos,
                            "contenido": textos, "_source_file": pdf["archivo"], "_evento_carga": pdf["evento"]})


if eventos:
    lista = ", ".join(f"'{e}'" for e in eventos)
    cismid = (spark.read.format("binaryFile").option("pathGlobFilter", "*.txt").load([f"{landing}/sismos/{ev}" for ev in eventos])
              .select(F.element_at(F.split("path", "/"), -2).alias("evento"), F.element_at(F.split("path", "/"), -1).alias("archivo"), "length", "content")
              .withColumn("archivo_estacion", F.regexp_replace("archivo", r"\.txt$", ""))
              .mapInPandas(decodificar, ESQUEMA_CISMID)
              .withColumn("_ingested_at", F.current_timestamp()))
    if spark.catalog.tableExists("bronze_cismid_registros"):
        cismid.write.mode("overwrite").option("replaceWhere", f"_evento_carga IN ({lista})").saveAsTable("bronze_cismid_registros")
    else:
        cismid.write.saveAsTable("bronze_cismid_registros")
    spark.table("bronze_cismid_registros").where(f"_evento_carga IN ({lista})").groupBy("_evento_carga").count().orderBy("_evento_carga").show(truncate=False)

## 5. Maestros
Datos de referencia que no cambian con cada sismo. Se cargan la primera vez o cuando `recargar_maestros = true`.

In [ ]:
# Los maestros se cargan la primera vez o si se pide recargarlos.
cargar_maestros = recargar_maestros or not spark.catalog.tableExists("bronze_open_buildings")
print("Cargar maestros:", cargar_maestros)

### Límites distritales

In [ ]:
# Límites distritales del INEI y de OSM, leídos con Spark.
if cargar_maestros:
    archivo_inei = MAESTROS / "limites_inei" / "peru_distrital_simple.geojson"
    escribir(con_metadatos(leer_geojson(archivo_inei), "limites_inei", archivo_inei.as_posix()), "bronze_limites_inei")
    archivo_comp = MAESTROS / "limites_inei" / "complemento_osm_la_punta_santa_anita.geojson"
    if archivo_comp.exists():
        escribir(con_metadatos(leer_geojson(archivo_comp), "osm", archivo_comp.as_posix()), "bronze_limites_complemento_osm")
    archivo_osm_dist = MAESTROS / "limites_osm" / "distritos_lima_callao_osm.geojson"
    if archivo_osm_dist.exists():
        od = leer_geojson(archivo_osm_dist).withColumn("ubigeo", F.col("ubigeo").cast("string")).withColumn("osm_id", F.col("osm_id").cast("long"))
        escribir(con_metadatos(od, "osm", archivo_osm_dist.as_posix()), "bronze_limites_osm_distritos")
    else:
        print("Falta", archivo_osm_dist.as_posix(), "(ver README): silver usará los límites simplificados del INEI")

### Vs30

In [ ]:
if cargar_maestros:
    ESQUEMA_VS30 = ("ciudad string, fila long, col long, lon double, lat double, vs30 double, "
                    "grid_origen_lon double, grid_origen_lat double, grid_res double")
    archivo_vs30 = (MAESTROS / "vs30" / "global_vs30.grd").as_posix()


    # Lee en los workers la ventana del ráster de Vs30 pedida.
    def ventanas_vs30(lotes):
        for pdf in lotes:
            for v in pdf.itertuples():
                with rasterio.open(v.ruta) as src:
                    ventana = from_bounds(v.xmin, v.ymin, v.xmax, v.ymax, transform=src.transform).round_offsets().round_lengths()
                    valores = src.read(1, window=ventana)
                    tr = src.window_transform(ventana)
                    origen_lon, origen_lat, res = src.transform.c, src.transform.f, src.res[0]
                filas, cols = np.indices(valores.shape)
                yield pd.DataFrame({"ciudad": v.ciudad, "fila": (filas + int(ventana.row_off)).ravel(), "col": (cols + int(ventana.col_off)).ravel(),
                                    "lon": (tr.c + (cols + 0.5) * tr.a).ravel(), "lat": (tr.f + (filas + 0.5) * tr.e).ravel(),
                                    "vs30": valores.ravel().astype("float64"), "grid_origen_lon": origen_lon, "grid_origen_lat": origen_lat, "grid_res": res})


    # Lanza la lectura de las ventanas de Vs30 en paralelo.
    def pixeles_vs30(cajas):
        filas = [(ciudad, archivo_vs30, *caja) for ciudad, caja in cajas.items()]
        return (spark.createDataFrame(filas, "ciudad string, ruta string, xmin double, ymin double, xmax double, ymax double")
                .repartition(len(filas)).mapInPandas(ventanas_vs30, ESQUEMA_VS30))


    MARGEN = 0.05
    caja = (spark.table("bronze_limites_inei").where(F.col("NOMBPROV").isin(PROVINCIAS_ESTUDIO) & F.col("geometry_wkt").isNotNull())
            .select(limites_wkt("geometry_wkt").alias("b"))
            .agg(F.min("b.xmin"), F.min("b.ymin"), F.max("b.xmax"), F.max("b.ymax")).first())
    px_lima = pixeles_vs30({"Lima": (caja[0] - MARGEN, caja[1] - MARGEN, caja[2] + MARGEN, caja[3] + MARGEN)}).drop("ciudad")
    escribir(con_metadatos(px_lima, "vs30_usgs", archivo_vs30), "bronze_vs30_pixeles")
    spark.table("bronze_vs30_pixeles").select(F.min("vs30"), F.median("vs30"), F.max("vs30")).show()

### Microzonificación

In [ ]:
if cargar_maestros:
    # Texto de cada página de un PDF.
    def paginas_pdf(ruta):
        lector = PdfReader(ruta)
        distrito = Path(ruta).stem.replace("microzonificacion_", "")
        return pd.DataFrame([{"distrito": distrito, "pagina": i, "n_paginas": len(lector.pages), "texto": pag.extract_text() or "", "_source_file": ruta}
                             for i, pag in enumerate(lector.pages, start=1)])


    paginas = leer_archivos(MAESTROS / "microzonificacion", "*.pdf", paginas_pdf, "distrito string, pagina long, n_paginas long, texto string, _source_file string")
    escribir(con_metadatos(paginas, "microzonificacion"), "bronze_microzonificacion_paginas")
    display(spark.table("bronze_microzonificacion_paginas").groupBy("distrito").agg(F.max("pagina").alias("paginas"), F.sum(F.length("texto")).alias("caracteres")))

### Open Buildings

In [ ]:
# Huellas de Open Buildings leídas con Spark.
if cargar_maestros:
    ob = (leer_csv(MAESTROS / "open_buildings", todo_texto=True, pathGlobFilter="*.{csv,csv.gz}", recursiveFileLookup=True)
          .select(*[F.col(c).cast("double") for c in ("latitude", "longitude", "area_in_meters", "confidence")], "geometry", "full_plus_code", "_source_file"))
    escribir(con_metadatos(ob, "open_buildings_v3"), "bronze_open_buildings")

### Open Buildings 2.5D

In [ ]:
if cargar_maestros:
    # Metadatos y estadísticas de una tesela de altura.
    def resumen_tesela(ruta):
        nombre = Path(ruta).stem
        with rasterio.open(ruta) as src:
            h = src.read(1, out_shape=(src.height // 8, src.width // 8))
            b = src.bounds
            valido = h[(h != src.nodata) & (h > 0)]
            return pd.DataFrame([{"tesela": nombre, "epsg": src.crs.to_epsg(), "res_m": src.res[0], "ancho": src.width, "alto": src.height,
                                  "xmin": b.left, "ymin": b.bottom, "xmax": b.right, "ymax": b.top, "nodata": float(src.nodata),
                                  "pct_px_con_edificio": round(100 * valido.size / h.size, 3),
                                  "altura_p50_m": float(np.median(valido)) if valido.size else None,
                                  "altura_p99_m": float(np.percentile(valido, 99)) if valido.size else None,
                                  "anio": int(re.search(r"_(\d{4})_\d\d_\d\d", nombre).group(1)), "_source_file": ruta}])


    teselas = leer_archivos(MAESTROS / "open_buildings_25d", "*.tif", resumen_tesela,
                            "tesela string, epsg long, res_m double, ancho long, alto long, xmin double, ymin double, xmax double, ymax double, nodata double, "
                            "pct_px_con_edificio double, altura_p50_m double, altura_p99_m double, anio long, _source_file string")
    escribir(con_metadatos(teselas, "open_buildings_25d"), "bronze_open_buildings_25d_teselas")

### STEAD (muestra de 10 000 trazas)

In [ ]:
if cargar_maestros:
    # Número de trazas y forma de una traza en el HDF5.
    def info_hdf5(ruta):
        with h5py.File(ruta, "r") as h:
            return pd.DataFrame([{"archivo": ruta, "n_trazas": len(h["data"]), "forma": str(h["data"][next(iter(h["data"]))].shape)}])


    carpeta_stead = MAESTROS / "stead"
    stead = leer_csv(carpeta_stead / "chunk2_muestra.csv", multiLine=True).withColumn("_archivo_hdf5", F.lit((carpeta_stead / "chunk2_muestra.hdf5").as_posix()))
    n_stead = escribir(con_metadatos(stead, "stead"), "bronze_stead_metadata")
    hdf5 = leer_archivos(carpeta_stead, "chunk2_muestra.hdf5", info_hdf5, "archivo string, n_trazas long, forma string").first()
    if hdf5:
        print(f"HDF5: {hdf5.n_trazas:,} trazas (CSV: {n_stead:,}) | {'coinciden' if hdf5.n_trazas == n_stead else 'NO coinciden'} | forma de una traza: {hdf5.forma}")

### Norma E.030

In [ ]:
# Tablas de la Norma E.030.
if cargar_maestros:
    for nombre in ["e030_zonas", "e030_factor_suelo", "e030_perfiles"]:
        escribir(con_metadatos(leer_csv(MAESTROS / "norma_e030" / f"{nombre}.csv"), "norma_e030"), f"bronze_{nombre}")

### Censo 2017

In [ ]:
# Material de paredes del Censo 2017 por distrito.
if cargar_maestros:
    archivo_censo = MAESTROS / "censo2017" / "paredes_distrito_lima_callao.csv"
    if archivo_censo.exists():
        escribir(con_metadatos(leer_csv(archivo_censo, como_texto=("ubigeo",)), "inei_censo2017_redatam"), "bronze_censo2017_paredes")
    else:
        print("Falta", archivo_censo.as_posix(), "(ver README)")

### OSM: número de pisos

In [ ]:
# Puntos de OSM con número de pisos.
if cargar_maestros:
    archivo_osm = MAESTROS / "osm_niveles" / "lima_building_levels.json"
    if archivo_osm.exists():
        esquema_osm = "elements array<struct<id: bigint, center: struct<lat: double, lon: double>, tags: map<string, string>>>"
        osm = (spark.read.schema(esquema_osm).option("multiLine", True).json(archivo_osm.as_posix())
               .select(F.explode("elements").alias("e")).where(F.col("e.center").isNotNull())
               .select(F.col("e.id").cast("string").alias("osm_id"), F.col("e.center.lon").alias("lon"), F.col("e.center.lat").alias("lat"),
                       F.col("e.tags")["building:levels"].alias("niveles_txt"))
               .withColumn("niveles", F.col("niveles_txt").try_cast("double")))
        escribir(con_metadatos(osm, "osm", archivo_osm.as_posix()), "bronze_osm_niveles")
    else:
        print("Falta", archivo_osm.as_posix(), "(ver README): los pisos salen solo de la altura 2.5D")

### Tainan

In [ ]:
if cargar_maestros:
    NS_XLSX = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"


    # Lee la primera hoja de un Excel sin depender de librerías extra.
    def hoja_xlsx(ruta):
        from xml.etree import ElementTree as ET
        with zipfile.ZipFile(ruta) as z:
            compartidas = ["".join(t.text or "" for t in si.iter(NS_XLSX + "t")) for si in ET.fromstring(z.read("xl/sharedStrings.xml")).findall(NS_XLSX + "si")]
            hoja = ET.fromstring(z.read("xl/worksheets/sheet1.xml"))
        filas = []
        for n, fila in enumerate(hoja.iter(NS_XLSX + "row"), start=1):
            celdas = {}
            for c in fila.findall(NS_XLSX + "c"):
                v = c.find(NS_XLSX + "v")
                if v is not None:
                    celdas[re.match(r"[A-Z]+", c.get("r")).group(0)] = compartidas[int(v.text)] if c.get("t") == "s" else v.text
            filas.append({"libro": Path(ruta).name, "fila": str(n), "celdas": json.dumps(celdas)})
        return pd.DataFrame(filas)


    # Lista los registros .txt que hay dentro del zip.
    def nombres_zip(lotes):
        for pdf in lotes:
            for ruta in pdf["ruta"]:
                with zipfile.ZipFile(ruta) as z:
                    yield pd.DataFrame({"ruta": ruta, "archivo": [n for n in z.namelist() if n.endswith(".txt") and not n.startswith("__MACOSX")]})


    # Lee el texto de los registros pedidos dentro del zip.
    def contenido_zip(lotes):
        for pdf in lotes:
            for ruta, g in pdf.groupby("ruta"):
                with zipfile.ZipFile(ruta) as z:
                    yield pd.DataFrame({"archivo": g["archivo"].to_numpy(), "contenido": [z.read(n).decode("latin-1") for n in g["archivo"]]})


    carpeta_tainan = MAESTROS / "tainan_2016"
    for tabla, archivo, codificacion in (("tainan_edificios", "edificios/parameters_taiwan_eq_20170828.csv", "UTF-8"),
                                         ("tainan_casos", "edificios/2016_Taiwan_Meinong_20170814.csv", "ISO-8859-1"),
                                         ("tainan_archivos", "edificios/1267_file_metadata.csv", "ISO-8859-1")):
        subir_crudo(leer_csv(carpeta_tainan / archivo, todo_texto=True, encoding=codificacion, multiLine=True), tabla, "purr_tainan")

    lista = (leer_archivos(carpeta_tainan / "ondas", "list_of_ground_motion*.xlsx", hoja_xlsx, "libro string, fila string, celdas string")
             .withColumn("celdas", F.from_json("celdas", "map<string, string>")))
    claves = sorted(r[0] for r in lista.select(F.explode(F.map_keys("celdas"))).distinct().collect())
    subir_crudo(lista.select("libro", "fila", *[F.col("celdas")[k].alias(k) for k in claves]), "tainan_lista_registros", "purr_tainan")

    zip_tainan = (carpeta_tainan / "ondas" / "Taiwan_CWB_ground_motion_data.zip").as_posix()
    registros_tainan = (spark.createDataFrame([(zip_tainan,)], "ruta string").mapInPandas(nombres_zip, "ruta string, archivo string")
                        .repartition(16).mapInPandas(contenido_zip, "archivo string, contenido string"))
    escribir(con_metadatos(registros_tainan, "purr_tainan", zip_tainan), "bronze_tainan_registros")

### Ecuador: Portoviejo y Manta

In [ ]:
# Edificios de Portoviejo y Manta, y registros de Manta.
if cargar_maestros:
    archivo_ecuador = MAESTROS / "portoviejo_2016" / "campo" / "parameters_priority_index_metadata.csv"
    subir_crudo(leer_csv(archivo_ecuador, todo_texto=True, multiLine=True), "ecuador_edificios", "purr_ecuador")
    manta = (spark.read.format("binaryFile").option("pathGlobFilter", "*.txt").option("recursiveFileLookup", "true")
             .load((MAESTROS / "manta_2016" / "ondas" / "datos").as_posix())
             .select(F.element_at(F.split("path", "/"), -1).alias("archivo"), F.decode("content", "ISO-8859-1").alias("contenido"),
                     sin_dbfs(F.col("path")).alias("_source_file")))
    escribir(con_metadatos(manta, "purr_ecuador"), "bronze_manta_registros")

### Vs30 de Tainan y Manta

In [ ]:
# Píxeles de Vs30 alrededor de Tainan y Manta.
if cargar_maestros:
    VENTANAS_CIUDAD = {"Tainan": (119.6, 22.2, 121.0, 23.8), "Manta": (-81.0, -1.2, -80.4, -0.7)}
    px_ciudades = pixeles_vs30(VENTANAS_CIUDAD).select("ciudad", "fila", "col", "lon", "lat", "vs30")
    escribir(con_metadatos(px_ciudades, "vs30_usgs", archivo_vs30), "bronze_vs30_pixeles_ciudades")

### BSSA14 y fragilidad GEM

In [ ]:
# Coeficientes BSSA14 y curvas de fragilidad GEM.
if cargar_maestros:
    subir_crudo(leer_csv(MAESTROS / "bssa14" / "bssa14_no_lineal.csv", todo_texto=True), "bssa14_no_lineal", "bssa14")
    gem = (spark.read.text((MAESTROS / "fragilidad_gem").as_posix(), wholetext=True, pathGlobFilter="*.csv")
           .select(F.col("_metadata.file_name").alias("archivo"), F.col("value").alias("contenido"), sin_dbfs(F.col("_metadata.file_path")).alias("_source_file")))
    escribir(con_metadatos(gem, "gem"), "bronze_fragilidad_gem")

## 6. Manifiesto del Volume

In [ ]:
# Inventario de los archivos del Volume.
manifiesto = (listar(landing, recursivo=True)
              .withColumn("archivo", F.regexp_extract("ruta", r"/landing/(.*)$", 1))
              .select("archivo", F.split("archivo", "/")[1].alias("carpeta"), F.col("length").alias("bytes"),
                      F.date_format("modificationTime", "yyyy-MM-dd'T'HH:mm:ss").alias("modificado")))
escribir(manifiesto, "bronze_landing_manifiesto")
display(spark.table("bronze_landing_manifiesto").groupBy("carpeta").agg(F.count("*").alias("archivos"), F.round(F.sum("bytes") / 1e9, 2).alias("gb")))

## 7. Verificación

In [ ]:
# Registros por sismo y tamaño de las tablas bronze.
spark.table("bronze_cismid_registros").groupBy("_evento_carga", "codificacion").count().orderBy("_evento_carga").show(truncate=False)
inv = inventario("bronze_")
total = inv.agg(F.count("*").alias("tablas"), F.sum("filas").alias("filas"), F.sum("mb").alias("mb")).first()
print(f"Tablas: {total.tablas} | filas: {total.filas:,} | tamaño: {total.mb:,.0f} MB")
display(inv)